# Анализ пересечения Мпемба в информационной геометрии нейросетей

## Цель анализа
Определить наличие статистически значимого эффекта Мпемба - явления, при котором "горячая" (дестабилизированная) инициализация нейронной сети сходится быстрее, чем "холодная" (близкая к эталону).

## Методология
1. **Усреднение по ансамблю**: Вычисление среднего KL и 95% доверительного интервала
2. **Детекция пересечения**: Нахождение первого момента $t_c$, где KL_hot < KL_cold
3. **Статистическая значимость**: t-критерий Уэлча (p < 0.01)
4. **Визуализация**: Графики с конвертами ошибок

In [ ]:
# Импорт библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from typing import Tuple, Dict
import os

# Настройка визуализации
sns.set_style("whitegrid")
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print("Библиотеки импортированы")

## 1. Загрузка данных

In [ ]:
# Путь к результатам
results_dir = '../results'
parquet_file = 'trajectories.parquet'
parquet_path = os.path.join(results_dir, parquet_file)

# Проверяем существование файла
if os.path.exists(parquet_path):
    df = pd.read_parquet(parquet_path)
    print(f"Загружено {len(df)} записей из {parquet_path}")
    print(f"\nКолонки: {list(df.columns)}")
    print(f"\nУникальные запуски: {df['run_id'].nunique()}")
    print(f"Типы инициализации: {df['init_type'].unique()}")
    print(f"\nПервые строки:")
    display(df.head())
else:
    print(f"Файл {parquet_path} не найден!")
    print("Сначала запустите эксперимент: python src/mpemba_runner.py --config configs/default.yaml")

## 2. Усреднение по ансамблю

In [ ]:
def compute_ensemble_statistics(
    df: pd.DataFrame,
    metric: str = 'kl_div_star'
) -> pd.DataFrame:
    """
    Вычисляет среднее и доверительные интервалы для каждого шага.
    
    Args:
        df: DataFrame с результатами
        metric: Название метрики
        
    Returns:
        DataFrame со статистикой по шагам
    """
    stats_list = []
    
    for init_type in ['cold', 'hot']:
        df_subset = df[df['init_type'] == init_type]
        
        # Группируем по шагу
        grouped = df_subset.groupby('step')[metric]
        
        for step, values in grouped:
            mean_val = values.mean()
            std_val = values.std()
            sem_val = values.sem()  # Standard Error of Mean
            
            # 95% доверительный интервал (через t-распределение)
            n = len(values)
            t_value = stats.t.ppf(0.975, df=n-1)
            ci_lower = mean_val - t_value * sem_val
            ci_upper = mean_val + t_value * sem_val
            
            stats_list.append({
                'step': step,
                'init_type': init_type,
                'mean': mean_val,
                'std': std_val,
                'sem': sem_val,
                'ci_lower': ci_lower,
                'ci_upper': ci_upper,
                'n_runs': n
            })
    
    return pd.DataFrame(stats_list)

# Вычисляем статистику
if 'df' in locals():
    df_stats = compute_ensemble_statistics(df, 'kl_div_star')
    print("Статистика вычислена")
    display(df_stats.head())

## 3. Детекция пересечения

In [ ]:
def detect_crossing_point(
    df_stats: pd.DataFrame,
    persistence_steps: int = 50
) -> Dict:
    """
    Находит первый момент пересечения кривых KL_hot и KL_cold.
    
    Args:
        df_stats: DataFrame со статистикой
        persistence_steps: Количество шагов для проверки устойчивости пересечения
        
    Returns:
        Словарь с информацией о точке пересечения
    """
    # Разделяем на hot и cold
    cold_stats = df_stats[df_stats['init_type'] == 'cold'].set_index('step').sort_index()
    hot_stats = df_stats[df_stats['init_type'] == 'hot'].set_index('step').sort_index()
    
    # Объединяем
    combined = pd.DataFrame({
        'cold_mean': cold_stats['mean'],
        'hot_mean': hot_stats['mean'],
        'cold_ci_lower': cold_stats['ci_lower'],
        'cold_ci_upper': cold_stats['ci_upper'],
        'hot_ci_lower': hot_stats['ci_lower'],
        'hot_ci_upper': hot_stats['ci_upper']
    })
    
    # Разница
    combined['diff'] = combined['cold_mean'] - combined['hot_mean']
    
    # Ищем первое устойчивое пересечение
    crossing_step = None
    consecutive_count = 0
    
    for step in combined.index:
        if combined.loc[step, 'diff'] > 0:  # hot < cold
            consecutive_count += 1
            if consecutive_count >= persistence_steps:
                crossing_step = step - persistence_steps + 1
                break
        else:
            consecutive_count = 0
    
    result = {
        'crossing_detected': crossing_step is not None,
        'crossing_step': crossing_step,
        'combined': combined
    }
    
    if crossing_step is not None:
        print(f"✓ Обнаружено пересечение на шаге {crossing_step}")
        print(f"  KL_cold = {combined.loc[crossing_step, 'cold_mean']:.6f}")
        print(f"  KL_hot  = {combined.loc[crossing_step, 'hot_mean']:.6f}")
        print(f"  Разница = {combined.loc[crossing_step, 'diff']:.6f}")
    else:
        print("✗ Пересечение НЕ обнаружено")
        print("  Кривые не пересекаются в течение всего обучения")
    
    return result

# Детектируем пересечение
if 'df_stats' in locals():
    crossing_result = detect_crossing_point(df_stats, persistence_steps=50)

## 4. Проверка статистической значимости

In [ ]:
def welch_t_test(
    df: pd.DataFrame,
    step: int,
    metric: str = 'kl_div_star'
) -> Dict:
    """
    Применяет t-критерий Уэлча для сравнения средних.
    
    H0: KL_cold - KL_hot <= 0
    H1: KL_cold - KL_hot > 0
    
    Args:
        df: DataFrame с результатами
        step: Номер шага для проверки
        metric: Название метрики
        
    Returns:
        Словарь с результатами теста
    """
    # Извлекаем значения для данного шага
    cold_values = df[(df['init_type'] == 'cold') & (df['step'] == step)][metric]
    hot_values = df[(df['init_type'] == 'hot') & (df['step'] == step)][metric]
    
    if len(cold_values) == 0 or len(hot_values) == 0:
        return {'error': 'Недостаточно данных'}
    
    # t-критерий Уэлча (односторонний)
    t_stat, p_value_two_sided = stats.ttest_ind(
        cold_values, hot_values, equal_var=False
    )
    
    # Односторонний p-value
    p_value_one_sided = p_value_two_sided / 2 if t_stat > 0 else 1 - p_value_two_sided / 2
    
    # Размеры выборок
    n_cold = len(cold_values)
    n_hot = len(hot_values)
    
    # Степени свободы (приближение Уэлча-Саттерсвейта)
    var_cold = cold_values.var()
    var_hot = hot_values.var()
    
    result = {
        'step': step,
        'n_cold': n_cold,
        'n_hot': n_hot,
        'mean_cold': cold_values.mean(),
        'mean_hot': hot_values.mean(),
        'std_cold': cold_values.std(),
        'std_hot': hot_values.std(),
        't_statistic': t_stat,
        'p_value': p_value_one_sided,
        'significant_001': p_value_one_sided < 0.01,
        'significant_005': p_value_one_sided < 0.05
    }
    
    print(f"Шаг {step}:")
    print(f"  N_cold = {n_cold}, N_hot = {n_hot}")
    print(f"  Mean_cold = {result['mean_cold']:.6f}, Mean_hot = {result['mean_hot']:.6f}")
    print(f"  t = {t_stat:.4f}")
    print(f"  p-value = {p_value_one_sided:.6f}")
    print(f"  Значимо (α=0.01): {'ДА' if result['significant_001'] else 'НЕТ'}")
    print(f"  Значимо (α=0.05): {'ДА' if result['significant_005'] else 'НЕТ'}")
    
    return result

# Проверяем значимость в точке пересечения
if 'crossing_result' in locals() and 'df' in locals():
    if crossing_result['crossing_detected']:
        print("\nПроверка значимости в точке пересечения:")
        print("="*60)
        t_test_result = welch_t_test(df, crossing_result['crossing_step'])
    else:
        print("\nПроверка значимости на последнем шаге:")
        print("="*60)
        last_step = df['step'].max()
        t_test_result = welch_t_test(df, last_step)

## 5. Визуализация

In [ ]:
def plot_kl_divergence(
    df_stats: pd.DataFrame,
    crossing_result: Dict,
    save_path: str = None
) -> None:
    """
    Строит график KL-дивергенции с конвертами ошибок.
    
    Args:
        df_stats: DataFrame со статистикой
        crossing_result: Результат детекции пересечения
        save_path: Путь для сохранения графика
    """
    fig, ax = plt.subplots(figsize=(14, 8))
    
    colors = {'cold': '#2196F3', 'hot': '#F44336'}
    
    for init_type in ['cold', 'hot']:
        subset = df_stats[df_stats['init_type'] == init_type]
        
        # Основная линия
        ax.plot(
            subset['step'], subset['mean'],
            label=f'{init_type.capitalize()} (Mean)',
            color=colors[init_type],
            linewidth=2.5
        )
        
        # Конверт ошибок (95% CI)
        ax.fill_between(
            subset['step'],
            subset['ci_lower'],
            subset['ci_upper'],
            alpha=0.2,
            color=colors[init_type],
            label=f'{init_type.capitalize()} (95% CI)'
        )
    
    # Линия пересечения
    if crossing_result['crossing_detected']:
        tc = crossing_result['crossing_step']
        ax.axvline(
            x=tc, color='green', linestyle='--', linewidth=2,
            label=f'Crossing point (t_c = {tc})'
        )
    
    ax.set_xlabel('Step (Iteration)', fontsize=14)
    ax.set_ylabel('KL Divergence to θ*', fontsize=14)
    ax.set_title('Mpemba Effect: KL Divergence vs Training Steps', fontsize=16)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"График сохранен: {save_path}")
    
    plt.show()

# Строим график
if 'df_stats' in locals() and 'crossing_result' in locals():
    plot_kl_divergence(
        df_stats,
        crossing_result,
        save_path=os.path.join(results_dir, 'kl_divergence_crossing.png')
    )

In [ ]:
def plot_fisher_trace(
    df_stats: pd.DataFrame,
    save_path: str = None
) -> None:
    """
    Строит графика следа матрицы Фишера.
    
    Args:
        df_stats: DataFrame со статистикой
        save_path: Путь для сохранения графика
    """
    fig, ax = plt.subplots(figsize=(14, 8))
    
    colors = {'cold': '#2196F3', 'hot': '#F44336'}
    
    for init_type in ['cold', 'hot']:
        subset = df_stats[df_stats['init_type'] == init_type]
        
        ax.plot(
            subset['step'], subset['mean'],
            label=f'{init_type.capitalize()}',
            color=colors[init_type],
            linewidth=2.5
        )
    
    ax.set_xlabel('Step (Iteration)', fontsize=14)
    ax.set_ylabel('Trace(Fisher Information Matrix)', fontsize=14)
    ax.set_title('Fisher Information Trace vs Training Steps', fontsize=16)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"График сохранен: {save_path}")
    
    plt.show()

# Строим график следа Фишера
if 'df_stats' in locals():
    df_fisher = compute_ensemble_statistics(df, 'trace_fisher')
    plot_fisher_trace(
        df_fisher,
        save_path=os.path.join(results_dir, 'fisher_trace.png')
    )

## 6. Финальный вердикт

In [ ]:
def generate_verdict(
    crossing_result: Dict,
    t_test_result: Dict
) -> str:
    """
    Формирует финальный вердикт о наличии эффекта Мпемба.
    
    Args:
        crossing_result: Результат детекции пересечения
        t_test_result: Результат t-теста
        
    Returns:
        Строка с вердиктом
    """
    print("="*80)
    print("ФИНАЛЬНЫЙ ВЕРДИКТ")
    print("="*80)
    
    if crossing_result['crossing_detected']:
        tc = crossing_result['crossing_step']
        
        if t_test_result.get('significant_001', False):
            verdict = f"""
✓ ЭФФЕКТ МПЕМБА ПОДТВЕРЖДЕН СТАТИСТИЧЕСКИ ЗНАЧИМО

Ключевые результаты:
  - Время пересечения: t_c = {tc} итераций
  - KL_cold(t_c) = {t_test_result['mean_cold']:.6f}
  - KL_hot(t_c)  = {t_test_result['mean_hot']:.6f}
  - t-статистика = {t_test_result['t_statistic']:.4f}
  - p-value = {t_test_result['p_value']:.6f} (p < 0.01)

Вывод: Горячая инициализация сходится БЫСТРЕЕ холодной,
что подтверждает наличие эффекта Мпемба в информационной геометрии.
"""
        elif t_test_result.get('significant_005', False):
            verdict = f"""
~ ЭФФЕКТ МПЕМБА ПОДТВЕРЖДЕН С УМЕРЕННОЙ ЗНАЧИМОСТЬЮ

Ключевые результаты:
  - Время пересечения: t_c = {tc} итераций
  - p-value = {t_test_result['p_value']:.6f} (0.01 < p < 0.05)

Вывод: Наблюдается тенденция к ускоренной сходимости горячей модели,
но требуется дополнительное подтверждение.
"""
        else:
            verdict = f"""
✗ ЭФФЕКТ МПЕМБА НЕ ПОДТВЕРЖДЕН

Ключевые результаты:
  - Пересечение обнаружено на шаге t_c = {tc}
  - p-value = {t_test_result['p_value']:.6f} (p >= 0.05)

Вывод: Разница между траекториями статистически не значима.
Эффект Мпемба НЕ обнаружен на требуемом уровне значимости.
"""
    else:
        verdict = """
✗ ЭФФЕКТ МПЕМБА НЕ ОБНАРУЖЕН

Кривые KL-дивергенции для горячей и холодной инициализации
НЕ пересекаются в течение всего обучения.

Вывод: Классический эффект Мпемба отсутствует в данной конфигурации.
"""
    
    print(verdict)
    return verdict

# Генерируем вердикт
if 'crossing_result' in locals() and 't_test_result' in locals():
    verdict = generate_verdict(crossing_result, t_test_result)

## 7. Дополнительный анализ: эволюция распределений

In [ ]:
# Распределение KL на разных этапах обучения
if 'df' in locals():
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Выбираем 4 ключевых момента
    max_step = df['step'].max()
    checkpoints = [
        max(1, max_step // 100),
        max_step // 10,
        max_step // 4,
        max_step // 2
    ]
    
    for ax, step in zip(axes.flat, checkpoints):
        for init_type in ['cold', 'hot']:
            values = df[(df['init_type'] == init_type) & (df['step'] == step)]['kl_div_star']
            if len(values) > 0:
                ax.hist(
                    values, bins=50, alpha=0.5, label=init_type,
                    color={'cold': '#2196F3', 'hot': '#F44336'}[init_type]
                )
        
        ax.set_title(f'Step {step}')
        ax.set_xlabel('KL Divergence')
        ax.legend()
    
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, 'kl_distributions.png'), dpi=300)
    plt.show()